In [ ]:
"""
============================================================
Simple PDF RAG Chatbot using Boeing AI Endpoints
------------------------------------------------------------
Features:
1. Reads a PDF
2. Splits it into chunks
3. Creates embeddings using BoeingEmbeddings
4. Stores them in ChromaDB
5. Retrieves relevant chunks
6. Answers questions using BoeingChatModel

Requirements:
pip install python-dotenv chromadb pypdf numpy
pip install python-dotenv chromadb pypdf numpy pydantic langchain-core requests httpx

Make sure the Boeing SDK packages are Available:
- boeing_chat_model
- boeing_embeddings

Create a .env file containing:

UDAL_PAT=your_pat_here
============================================================
"""

import os
import uuid
import textwrap

from dotenv import load_dotenv
from pypdf import PdfReader
import chromadb

from boeing_chat_model import BoeingChatModel
from boeing_embeddings import BoeingEmbeddings

# ============================================================
# LOAD ENVIRONMENT VARIABLES
# ============================================================

load_dotenv()

UDAL_PAT = os.getenv("UDAL_PAT")

if not UDAL_PAT:
    raise ValueError("UDAL_PAT not found in .env file")

# ============================================================
# BOEING CLIENTS
# ============================================================

chat_client = BoeingChatModel(
    udal_pat=UDAL_PAT,
    model="gpt-4o-mini",
    max_tokens=500,
    temperature=0.2,
)

embedding_client = BoeingEmbeddings(
    udal_pat=UDAL_PAT,
    model="text-embedding-3-small",
)

# ============================================================
# CHROMADB SETUP
# ============================================================

chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection = chroma_client.get_or_create_collection(
    name="simple_rag"
)

# ============================================================
# LOAD PDF
# ============================================================

def load_pdf(pdf_path):
    """
    Read a PDF and return all extracted text.
    """

    reader = PdfReader(pdf_path)

    text = ""

    for page in reader.pages:

        extracted = page.extract_text()

        if extracted:
            text += extracted + "\n"

    return text


# ============================================================
# TEXT CHUNKING
# ============================================================

def chunk_text(text, chunk_size=800, overlap=200):
    """
    Split text into overlapping chunks.
    """

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks


# ============================================================
# EMBEDDINGS
# ============================================================

def get_embedding(text):
    """
    Generate embedding using Boeing Embeddings.
    """

    embedding = embedding_client.embed_query(text)

    return list(embedding)


# ============================================================
# STORE CHUNKS
# ============================================================

def add_chunks_to_chroma(chunks):
    """
    Store document chunks in ChromaDB.
    """

    print("\nCreating embeddings...")

    for chunk in chunks:

        embedding = get_embedding(chunk)

        collection.add(
            ids=[str(uuid.uuid4())],
            documents=[chunk],
            embeddings=[embedding],
        )

    print(f"\nStored {len(chunks)} chunks successfully.")


# ============================================================
# RETRIEVE DOCUMENTS
# ============================================================

def retrieve(query, top_k=3):
    """
    Retrieve top matching chunks with similarity scores.
    """

    query_embedding = get_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "distances"],
    )

    return results


# ============================================================
# ASK QUESTION
# ============================================================

def ask(query):
    """
    Ask Boeing GPT using retrieved context only.
    """

    results = retrieve(query)

    docs = results["documents"][0]
    distances = results["distances"][0]

    print("\n" + "=" * 100)
    print("RETRIEVED DOCUMENTS")
    print("=" * 100)

    for i, (doc, distance) in enumerate(zip(docs, distances), start=1):

        # Convert distance to an approximate similarity score
        similarity = 1 - distance

        print(f"\n--- Retrieved Chunk {i} ---")
        print(f"Distance   : {distance:.4f}")
        print(f"Similarity : {similarity:.4f}\n")

        print(textwrap.fill(doc, width=100))
        print("-" * 100)

    context = "\n\n".join(docs)

    prompt = f"""
You are a helpful RAG assistant.

Answer ONLY using the provided context.

If the answer is not present, respond exactly:

"I could not find the answer in the document."

Context:
{context}

Question:
{query}
"""

    messages = [
        {
            "role": "system",
            "content": "Answer only using retrieved documents."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    response = chat_client.invoke(messages)

    return response.content

# ============================================================
# BUILD VECTOR DATABASE
# ============================================================

def build_rag(pdf_path):
    """
    Read PDF, split into chunks and build vector DB.
    """

    print("\nLoading PDF...")

    text = load_pdf(pdf_path)

    print("PDF Loaded.")

    print("\nChunking document...")

    chunks = chunk_text(
        text,
        chunk_size=800,
        overlap=200,
    )

    print(f"Created {len(chunks)} chunks.")

    if collection.count() == 0:

        add_chunks_to_chroma(chunks)

    else:

        print(
            "ChromaDB already contains data. "
            "Skipping embedding generation."
        )


# ============================================================
# CHAT LOOP
# ============================================================

def chatbot():
    """
    Interactive chatbot.
    """

    print("\n=======================================")
    print("      PDF RAG CHATBOT (BOEING AI)")
    print("=======================================")
    print("Type 'exit' to quit.\n")

    while True:

        query = input("You: ").strip()

        if query.lower() == "exit":

            print("\nBot: Goodbye!")

            break

        answer = ask(query)

        print("\nBot:\n")

        print(textwrap.fill(answer, width=100))

        print("\n" + "=" * 100 + "\n")


# ============================================================
# MAIN
# ============================================================

def main():

    pdf_path = input("Enter PDF path: ").strip()

    if not os.path.exists(pdf_path):

        print("\nPDF not found.")

        return

    build_rag(pdf_path)

    chatbot()


# ============================================================
# ENTRY POINT
# ============================================================

if __name__ == "__main__":
    main()